# Giftmaxxing — Recommendation Lab

**One notebook, five questions:**

1. How much preference data do we actually have? (swipe labels)
2. Can a 10-feature logistic model beat the cosine similarity production already uses?
3. Is any difference real, or noise at our sample size?
4. Which ranker is serving which item? (the new attribution logging)
5. What would two *simulated users* actually be recommended?

---

## Read this before you run anything

**This notebook costs money while it is running.** `ml.t3.medium` ≈ **$0.058/hr**.
It does not stop by itself.

```bash
aws sagemaker stop-notebook-instance \
  --notebook-instance-name giftmaxxing-reco-lab --region us-east-1
```

**Kernel:** `conda_python3`.

**It reads production data read-only.** Nothing here writes to DynamoDB, S3
Vectors or any endpoint. The IAM policy attached to this notebook's role
(`giftmaxxing-notebook-data-read`) grants Scan/Query/GetItem only.

**The honest expectation:** as of the audit in `docs/data-inventory.md` we have
~70 positive swipe labels from 19 people. That is enough to *fit* ten weights
and nowhere near enough to *prove* they beat cosine. Expect a wide confidence
interval, and treat a negative result as a real result.

## 0 · Setup

Pull the repo's ML modules so the notebook and the exporter share one implementation of the features — a feature computed here must mean what it means in production.

In [ ]:
# The lifecycle config syncs infra/ml into ~/SageMaker/giftmaxxing-ml on start.
# If it is missing (instance started before the config was attached), this
# pulls it straight from S3 — no git auth needed.
import os, sys, subprocess

ML = "/home/ec2-user/SageMaker/giftmaxxing-ml"
if not os.path.isdir(ML):
    os.makedirs(ML, exist_ok=True)
    subprocess.run(["aws", "s3", "sync", "s3://giftmaxxing-dev-ml/notebook/", ML,
                    "--region", "us-east-1"], check=True)

sys.path.insert(0, ML)
os.chdir(ML)
os.makedirs("data", exist_ok=True); os.makedirs("model", exist_ok=True)
print("working dir:", os.getcwd())
print("modules   :", sorted(f for f in os.listdir('.') if f.endswith('.py')))

In [ ]:
import boto3, numpy as np
from collections import Counter, defaultdict

REGION = "us-east-1"
PREFIX = "giftmaxxing-dev"
ddb = boto3.client("dynamodb", region_name=REGION)
s3v = boto3.client("s3vectors", region_name=REGION)

print("identity:", boto3.client("sts").get_caller_identity()["Account"])

## 1 · How much preference data do we have?

Two sources, both *explicit per-item judgements* — far better than the
"dwelled ≥60s" proxy the MTL model was trained on (63 positives, two of its
three heads at 2 and 3 positives).

| Source | What it is |
|---|---|
| `challenges` RESP rows | a friend swiping a deck built *for them* — the person the gift is for |
| `analytics` swipe_right/left | the account owner swiping their own deck |

In [ ]:
from export_swipe_data import collect_labels

rows = collect_labels()          # [(user, post_id, y, ts, origin)]
pos = sum(r[2] for r in rows)
print(f"{len(rows)} swipe labels — {pos} yes / {len(rows)-pos} no")
print(f"{len(set(r[0] for r in rows))} distinct swipers")
print("by origin:", dict(Counter(r[4] for r in rows)))
print()
print("For comparison, the MTL model trained on 63 positives across 47 users,")
print("with one user contributing 24% of all rows.")

## 2 · Build the training set

Ten features, each hand-checkable. The full list lives in `swipe_model.FEATURE_NAMES`.

**Chronology is enforced.** Each example's user profile is built only from that
user's *earlier* swipes. Building profiles from a user's whole history leaks the
answer into the features and produces an offline number that means nothing.

In [ ]:
!python3 export_swipe_data.py --out data/swipes.npz

d = np.load("data/swipes.npz", allow_pickle=True)
X, Y, groups = d["X"], d["Y"], d["groups"]
from swipe_model import FEATURE_NAMES
print(f"\n{X.shape[0]} examples × {X.shape[1]} features · {int(Y.sum())} positive")
print("features:", FEATURE_NAMES)

## 3 · Train, and compare against what production already does

The bar is **not** "does it train". Logistic regression always trains. The bar is
**does it beat `cos_user_item` alone**, which is what the vector path serves today
for free.

Evaluation is **leave-one-user-out**. A random split would put the same swiper on
both sides and score the model on memorising one person.

In [ ]:
from swipe_model import LogisticSwipeModel, auc
from train_swipe import loocv

i_cos = FEATURE_NAMES.index("cos_user_item")
base_cos = auc(Y, X[:, i_cos])
oof = loocv(X, Y, groups); ok = ~np.isnan(oof)
model_auc = auc(Y[ok], oof[ok])

print("AUC  (0.5 = coin flip)")
print(f"  cosine only  (production today)  {base_cos:.3f}")
print(f"  logistic, leave-one-user-out     {model_auc:.3f}")
print(f"  lift                             {model_auc - base_cos:+.3f}")

### 3b · Is that lift real?

**This is the cell that decides whether anything ships.** A point estimate at
this sample size is meaningless without an interval.

In [ ]:
rng = np.random.default_rng(0)
idx = np.where(ok)[0]
deltas = []
for _ in range(2000):
    b = rng.choice(idx, size=len(idx), replace=True)
    a1, a0 = auc(Y[b], oof[b]), auc(Y[b], X[b, i_cos])
    if not (np.isnan(a1) or np.isnan(a0)):
        deltas.append(a1 - a0)
deltas = np.array(deltas)
lo, hi = np.percentile(deltas, [2.5, 97.5])
print(f"bootstrap ({len(deltas)} resamples)")
print(f"  mean lift    {deltas.mean():+.3f}")
print(f"  95% CI       [{lo:+.3f}, {hi:+.3f}]")
print(f"  P(lift > 0)  {(deltas > 0).mean():.2f}")
print()
print("  VERDICT:", "real — CI excludes zero" if lo > 0 else
      "NOT SIGNIFICANT — cannot distinguish from cosine at this sample size")

### 3c · What did it learn?

Standardized weights are directly comparable. Read the *signs* — a wrong sign is a bug you can see, which is the whole reason for choosing ten features over two million parameters.

In [ ]:
final = LogisticSwipeModel(l2=1.0).fit(X, Y)
for k, v in sorted(final.weights().items(), key=lambda kv: -abs(kv[1])):
    print(f"  {k:20} {v:+.3f}  {'#' * int(min(abs(v), 2) * 15)}")

## 4 · Which ranker is serving what?

The app now stamps every feed impression with `servedBy`:

| value | meaning |
|---|---|
| `server:facet+ondevice` | `GET /feed` (scorePost) then re-ranked on device |
| `server:vector` | `/recommendations` cosine, placed directly |
| `server:vector+mtl` | the trained MTL model |

`serverRank` vs final `position` gives `rankDelta` — how far the on-device ranker
moved an item. **This is what makes the on-device re-rank measurable instead of
merely logged.** Until a build carrying this ships and testers use it, expect
these to be empty.

In [ ]:
from export_swipe_data import scan_all, S, N

ev = scan_all(f"{PREFIX}-analytics")
served, deltas_by_src, outcomes = Counter(), defaultdict(list), defaultdict(lambda: [0, 0])
dwell_by_post = {}
for e in ev:
    if S(e, "type") == "feed_dwell":
        dwell_by_post[S(e, "postId", "")] = N(e, "dwellMs", 0)
for e in ev:
    sb = S(e, "servedBy")
    if not sb:
        continue
    served[sb] += 1
    rd = N(e, "rankDelta")
    if rd is not None:
        deltas_by_src[sb].append(rd)
    d = dwell_by_post.get(S(e, "postId", ""), 0)
    outcomes[sb][0] += 1
    outcomes[sb][1] += 1 if d >= 3000 else 0

if not served:
    print("No attributed impressions yet — ship a build with the servedBy logging first.")
    print("Everything below this cell still runs; this section fills in over time.")
else:
    print(f"{'servedBy':28} {'impr':>6} {'3s+ dwell':>10} {'rate':>7} {'mean rankDelta':>15}")
    for k, n in served.most_common():
        tot, good = outcomes[k]
        md = np.mean(deltas_by_src[k]) if deltas_by_src[k] else float("nan")
        print(f"  {k:26} {n:>6} {good:>10} {good/max(tot,1):>6.1%} {md:>15.2f}")

## 5 · Two simulated users

We have 13 signed-in users, so we cannot A/B anything. What we *can* do is check
the model behaves sensibly for two clearly different people — a sanity test, not
evidence.

The personas are grounded in the real Reddit corpus (`web/lib/reddit-gifts.json`,
2,432 posts): **r/BuyItForLife** (durable, practical, higher price tolerance) vs
**r/somethingimade + r/GiftIdeas** (handmade, personal, lower price). Median
price in that corpus is \$50, p25 \$16, p75 \$110 — used to set their price bands.

Each persona is turned into a taste centroid by embedding seed products from the
live catalog, exactly as a real user's centroid is built.

In [ ]:
PERSONAS = {
    "buyitforlife_ben": {
        "blurb": "r/BuyItForLife — durable, practical, pays more for things that last",
        "seed_terms": ["cast iron skillet", "leather wallet", "wool blanket",
                        "stainless steel water bottle", "mechanical watch"],
        "median_price": 110.0,
        "categories": {"kitchen", "home", "fashion"},
    },
    "handmade_hana": {
        "blurb": "r/somethingimade + r/GiftIdeas — handmade, personal, small-batch",
        "seed_terms": ["handmade ceramic mug", "personalized photo frame",
                        "embroidered tote", "scented soy candle", "custom name necklace"],
        "median_price": 30.0,
        "categories": {"home", "jewelry", "art"},
    },
}
for k, v in PERSONAS.items():
    print(f"{k:20} {v['blurb']}")

In [ ]:
# Build each persona's taste centroid by embedding its seed terms with the SAME
# Titan model the catalog was embedded with, then averaging - identical maths to
# getCentroid() in handler.mjs.
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

def embed_text(text):
    r = bedrock.invoke_model(
        modelId="amazon.titan-embed-image-v1",
        contentType="application/json", accept="application/json",
        body=json.dumps({"inputText": text[:200],
                          "embeddingConfig": {"outputEmbeddingLength": 1024}}))
    return np.array(json.loads(r["body"].read())["embedding"], dtype=np.float32)

for name, p in PERSONAS.items():
    p["centroid"] = np.mean([embed_text(t) for t in p["seed_terms"]], axis=0)
    print(f"{name}: centroid built from {len(p['seed_terms'])} seeds")

In [ ]:
# What does each persona get recommended? kNN against the live catalog, then
# re-scored by the trained model - the two-stage shape a unified system would use.
from swipe_model import UserProfile, build_features

def recommend(persona, k=8):
    r = s3v.query_vectors(vectorBucketName=f"{PREFIX}-vectors", indexName="pins",
                           topK=60, queryVector={"float32": persona["centroid"].tolist()},
                           returnMetadata=True, returnDistance=True)
    prof = UserProfile()
    prof.pos_centroid = persona["centroid"]
    prof.median_price = persona["median_price"]
    prof.liked_categories = persona["categories"]
    out = []
    for v in r.get("vectors", []):
        m = v.get("metadata") or {}
        meta = {"price": float(m.get("price") or 0), "category": m.get("category", ""),
                "merchant": m.get("domain", ""), "domain": m.get("domain", ""),
                "likes": 0, "has_gallery": False, "has_story": False,
                "is_retailer": not str(m.get("source", "")).startswith("Pinterest")}
        vec = np.array(v["data"]["float32"], dtype=np.float32) if v.get("data") else None
        feats = build_features(vec if vec is not None else persona["centroid"], meta, prof)
        out.append((float(final.predict_proba(feats.reshape(1, -1))[0]),
                    1 - (v.get("distance") or 1), m.get("title", "?")[:52],
                    meta["price"], meta["category"]))
    out.sort(key=lambda t: -t[0])
    return out[:k]

for name, p in PERSONAS.items():
    print(f"\n=== {name} — {p['blurb']}")
    print(f"{'model':>6} {'cos':>6}  {'$':>6}  {'category':12} title")
    for s, cos, title, price, cat in recommend(p):
        print(f"{s:>6.3f} {cos:>6.3f}  {price:>6.0f}  {str(cat)[:12]:12} {title}")

### 5b · Do the two personas actually get different things?

If the overlap is high, the system is recommending *popular*, not *personal* — a
failure mode worth catching.

In [ ]:
a = {t for _, _, t, _, _ in recommend(PERSONAS["buyitforlife_ben"], k=20)}
b = {t for _, _, t, _, _ in recommend(PERSONAS["handmade_hana"], k=20)}
overlap = len(a & b) / max(len(a | b), 1)
print(f"top-20 Jaccard overlap: {overlap:.0%}")
print("shared:", list(a & b)[:5] or "none")
print()
print("low overlap  -> the centroid is doing real work")
print("high overlap -> we are serving popularity, not taste")

## 6 · Summary

Run this last. It prints the decision, not the vibes.

In [ ]:
print("=" * 64)
print(f"labels            {len(Y)} examples, {int(Y.sum())} positive, {len(set(groups))} users")
print(f"cosine baseline   AUC {base_cos:.3f}")
print(f"logistic (LOUCV)  AUC {model_auc:.3f}   lift {model_auc-base_cos:+.3f}")
print(f"95% CI            [{lo:+.3f}, {hi:+.3f}]")
print(f"persona overlap   {overlap:.0%}")
print("=" * 64)
if lo > 0:
    print("SHIP: the model beats cosine on held-out users.")
else:
    print("DO NOT SHIP over cosine. The interval includes zero — at this")
    print("sample size the two are indistinguishable. Collect more labels:")
    print("  • every swipe deck completed adds ~14 labels")
    print("  • the servedBy logging (section 4) is what makes the NEXT")
    print("    comparison attributable")
    print("  • ~10^3 positives is the rough bar; we have ~70")
print("=" * 64)
print("\nSTOP THIS NOTEBOOK when you are done:")
print("  aws sagemaker stop-notebook-instance \\")
print("    --notebook-instance-name giftmaxxing-reco-lab --region us-east-1")